# 🤖 Build Your First AI Agent (with Skills)

**What you're about to build:** an agent that doesn't just answer questions — it picks the right *skill* for the job and uses it.

**Why skills?** A skill is just a folder with a markdown file that says "here's how to do X well." The agent reads the titles of skills it has available, picks one that matches your question, loads the instructions, and follows them. You can write skills in plain English. No JSON, no weird syntax.

This is exactly how real AI systems work today — Claude, Copilot, Cursor — they all read the same format. By the end of this hour you'll understand it from the inside.

You're running inside a **GitHub Codespace** — a free cloud computer with everything pre-installed. No setup needed. Let's go.

## ⚙️ First: connect a Python kernel

Before you run any code, the notebook needs a Python "engine" attached.

1. Look at the **top-right corner of this notebook** — you'll see a button that says **Select Kernel** (or shows a Python version if one is already picked).
2. Click it → choose **Python Environments** → pick the recommended **Python 3** option.
3. If a popup asks to install the Python or Jupyter extension, click **Install** and wait a few seconds.

You'll know it worked when the top-right shows something like `Python 3.x.x` instead of "Select Kernel." Then keep going. ⬇️


## ▶️ How to run a cell

Every gray box below is a **cell**. To run one:

- **Click the ▶️ play button** on the left edge of the cell, OR
- **Click inside the cell and press `Shift` + `Enter`**

**How to tell it ran:** the `[ ]` on the left becomes `[*]` while it's running, then `[1]` (or some number) when it finishes. You'll usually see output appear right under the cell.

**Always run cells top-to-bottom, in order.** Later cells depend on stuff defined earlier. If you skip around, things break.


## 🔑 Step 1: Get + paste your GitHub Models token

You need a **GitHub Models token** — it's the password the notebook uses to talk to the AI model. It looks like `github_pat_...` or `ghp_...`.

### 👉 If you already made one during pre-flight
Skip to "Run the cell below" at the bottom of this section.

### 👉 If you don't have one yet (most people — that's fine!)
Do this now, it takes ~2 minutes:

1. Open **[the GPT-4o mini playground](https://github.com/marketplace/models/azure-openai/gpt-4o-mini/playground)** in a new tab
2. Sign in to GitHub if it asks. If it's your first time, accept the GitHub Models terms.
3. Click **Use this model** (top right of the playground)
4. In the dialog that opens, click **Get developer key** (or "Create personal access token")
5. GitHub takes you to a pre-filled token form — **don't change anything**. Scroll down and click **Generate token**.
6. **COPY THE TOKEN NOW** — you'll only see it once. It starts with `github_pat_` or `ghp_`.

> ⚠️ **Got a "permission denied" error trying a plain PAT from `github.com/settings/tokens`?** That's expected — use the playground link above instead. It auto-scopes the token correctly for GitHub Models.

### ▶️ Run the cell below

> 🚨 **READ THIS FIRST — this trips up almost everyone:**
> 
> When you run the next cell, a **text input box appears at the very TOP of the VS Code window** — not in the cell, not on the page. Look up.
> 
> 1. **Paste your token into that top box** (Ctrl+V or Cmd+V)
> 2. **Press Enter**
> 
> If you paste your token inside the code instead, it won't work and your token will be visible to anyone who sees your screen. The popup at the top is the only place that's safe.
> 
> **If you don't see the popup:** click back into this notebook tab (sometimes VS Code shows the popup behind other things). Still nothing? Re-run the cell.

**Why a popup instead of just typing in the code?** Because tokens are secrets. If you paste one into code and push to GitHub, bots find it in seconds and rack up API calls on your account. The popup keeps it out of the file.


In [1]:
import os
from getpass import getpass

os.environ["GITHUB_TOKEN"] = getpass("Paste your GitHub Models token (github_pat_... or ghp_...): ")
print("✅ Token loaded")


✅ Token loaded


## 🔌 Step 2: Connect to the model

GitHub Models gives you free access to GPT-4o-mini, Phi, Llama, and others. Same API, different models — change one string and you're running a different brain.

The `openai` library is already installed in your Codespace.

In [2]:
from openai import OpenAI

# Same pattern as the official GitHub Models sample — just in Python.
# https://github.com/marketplace/models  → pick a model → "Code" tab
token = os.environ["GITHUB_TOKEN"]

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=token,
)

MODEL = "openai/gpt-4o-mini"

# Quick sanity check — is it alive?
response = client.chat.completions.create(
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "Say hi in exactly 5 words."},
    ],
    model=MODEL,
    temperature=1,
    max_tokens=4096,
    top_p=1,
)
print(response.choices[0].message.content)


Hello! How are you today?


## 📚 What's a skill? (the key idea)

A **skill** is a folder with instructions inside. That's it.

The agent gets a list of skill *names and short descriptions* — like a table of contents. When you ask a question, the agent:

1. Reads the skill list
2. Picks the most relevant one
3. **Loads the full instructions** for that skill
4. Follows them to answer you

Here's the magic: **the agent doesn't see the full instructions until it decides to load them.** This is called *progressive disclosure*. It keeps the agent focused and lets you have dozens of skills without overwhelming the model.

One important thing: **the model doesn't load skills on its own. YOUR CODE does.** The model says "please load `haiku-writer`" and your program decides whether to honor that. That's the safety boundary — the agent can only use skills you've given it.

## 🧰 Step 3: Look at the real skills — they're already on disk

**Open the file tree on the left side of VS Code.** Expand `.github/skills/`.

You'll see folders like this:

```
.github/skills/
├── caveman/
│   └── SKILL.md
├── haiku-writer/
│   └── SKILL.md
├── math-tutor/
│   └── SKILL.md
└── pirate-translator/
    └── SKILL.md
```

**Click `haiku-writer/SKILL.md` and read it.** That's what a real skill looks like. Plain markdown. Plain English. No code.

In a training-wheels version of this lesson, we'd hard-code these skills as Python strings inside a dictionary in this notebook. Instead, we're doing it the **real** way — reading them straight from disk, in the production format (`.github/skills/<name>/SKILL.md`) that every major AI tool understands.

## 🗂️ Step 4: Load the skills from disk

This function reads every `SKILL.md` file in `.github/skills/`, parses the YAML frontmatter to get the name and description, and keeps the body as the instructions.

In [3]:
import os
import re
from pathlib import Path

SKILLS_DIR = Path(".github/skills")

def load_skills_from_disk():
    """Read every SKILL.md file and return a dict of skills."""
    skills = {}
    for skill_md in SKILLS_DIR.glob("*/SKILL.md"):
        content = skill_md.read_text()
        # Parse YAML frontmatter (between --- markers)
        match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", content, re.DOTALL)
        if not match:
            continue
        frontmatter, body = match.groups()

        # Pull out name and description from the frontmatter
        name_match = re.search(r"^name:\s*(.+)$", frontmatter, re.MULTILINE)
        desc_match = re.search(r"^description:\s*(.+)$", frontmatter, re.MULTILINE)
        if not (name_match and desc_match):
            continue

        name = name_match.group(1).strip()
        description = desc_match.group(1).strip()

        skills[name] = {
            "description": description,
            "instructions": body.strip(),
            "path": str(skill_md),
        }
    return skills

SKILLS = load_skills_from_disk()
print(f"✅ Loaded {len(SKILLS)} skills from disk:\n")
for name, skill in SKILLS.items():
    print(f"  📖 {name}")
    print(f"     {skill['description'][:80]}...")
    print(f"     ({skill['path']})")
    print()

✅ Loaded 4 skills from disk:

  📖 caveman
     Ultra-compressed communication mode. Cuts token usage by ~75% by speaking like a...
     (.github/skills/caveman/SKILL.md)

  📖 pirate-translator
     Rewrites any text in pirate-speak. Use when the user asks to "translate to pirat...
     (.github/skills/pirate-translator/SKILL.md)

  📖 haiku-writer
     Writes a haiku (5-7-5 syllable poem) about a given topic. Use when the user asks...
     (.github/skills/haiku-writer/SKILL.md)

  📖 math-tutor
     Explains a math problem step by step, like a patient tutor. Use for any math que...
     (.github/skills/math-tutor/SKILL.md)



## 📝 Step 5: Give the agent a menu + the `load_skill` tool

The agent needs to know what skills exist before it can pick one. We give it just the **names and descriptions** — not the full instructions. That's the progressive disclosure part.

We do this by giving the agent ONE tool: `load_skill`. When the agent calls it, we return the skill's full instructions from the corresponding file on disk.

In [4]:
def build_skill_menu():
    lines = ["Available skills:"]
    for name, skill in SKILLS.items():
        lines.append(f"- {name}: {skill['description']}")
    return "\n".join(lines)

print(build_skill_menu())

Available skills:
- caveman: Ultra-compressed communication mode. Cuts token usage by ~75% by speaking like a smart caveman while keeping full technical accuracy. Use when the user says "caveman mode", "talk like caveman", "less tokens", or "be brief".
- pirate-translator: Rewrites any text in pirate-speak. Use when the user asks to "translate to pirate", "make this piratey", or similar.
- haiku-writer: Writes a haiku (5-7-5 syllable poem) about a given topic. Use when the user asks for a haiku, a short poem, or mentions "5-7-5".
- math-tutor: Explains a math problem step by step, like a patient tutor. Use for any math question where the user seems to want to learn the process, not just get an answer.


In [5]:
# The agent has ONE tool: load_skill.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "load_skill",
            "description": "Load the full instructions for a skill by name. Call this when a skill looks relevant to the user's request.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skill_name": {
                        "type": "string",
                        "description": "The exact name of the skill to load, e.g. 'haiku-writer'"
                    }
                },
                "required": ["skill_name"],
            },
        },
    },
]

def load_skill(skill_name: str) -> str:
    """Return the full instructions for a skill by reading it from disk."""
    if skill_name not in SKILLS:
        return f"Error: no skill named '{skill_name}'. Available: {list(SKILLS.keys())}"
    return SKILLS[skill_name]["instructions"]

print("✅ load_skill tool ready")

✅ load_skill tool ready


## 🔄 Step 6: The agent loop

Same loop pattern you'd see in any real agent. The only tool is `load_skill`, but the agent uses it to pull in *any* of the skills on disk.

Read this carefully — once you understand these ~25 lines, you understand agents.

In [6]:
import json

def run_agent(user_message: str, max_steps: int = 5) -> str:
    system_prompt = f"""You are an agent that uses skills to help users.

{build_skill_menu()}

When the user asks something, look at the skill list. If one matches, call load_skill to get its instructions, then follow them. If no skill matches, just answer normally."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message
        messages.append(msg)

        # If the model didn't ask to load a skill, it's done. Return its answer.
        if not msg.tool_calls:
            return msg.content

        # The model wants to load one or more skills. Do it.
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            skill_name = args["skill_name"]
            print(f"  📖 Agent loaded skill: {skill_name}")

            instructions = load_skill(skill_name)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": instructions,
            })

    return "(agent hit step limit)"

print("✅ Agent defined")

✅ Agent defined


## 🚀 Step 7: Run it!

Watch what the agent does. Notice how it picks a skill on its own.

In [7]:
prompt = "Summarize War and Peace like a caveman"

print(f"👤 USER: {prompt}\n")
answer = run_agent(prompt)
print(f"\n🤖 AGENT: {answer}")

👤 USER: Summarize War and Peace like a caveman

  📖 Agent loaded skill: caveman

🤖 AGENT: "Big story war, peace. Many characters: Pierre, Natasha, Andrei. Love, loss, battles. France, Russia fight. Fate decide lives. Learn life, honor, choices. Peace come, but pain stay."


### Try different prompts

Change the prompt and run again. Watch which skill the agent picks — and what happens when no skill fits.

In [ ]:
prompt = "Translate this to pirate: I love programming and coffee."
# prompt = "Can you help me solve 3x + 7 = 22?"
# prompt = "What year did the Roman Empire fall?"  # no skill matches — agent should just answer
# prompt = "Write a haiku about my math homework."  # what will it pick?

print(f"👤 USER: {prompt}\n")
answer = run_agent(prompt)
print(f"\n🤖 AGENT: {answer}")

## 🪨 The big reveal: Copilot reads the same files

Here's the thing that makes this whole session click.

**Your agent reads `.github/skills/*.md`. So does GitHub Copilot. So does Claude Code. So does Cursor.**

`.github/skills/` is the industry-standard location for AI skills. It's not a convention we made up for this class — it's the real spec that GitHub published in December 2025 and that every major AI tool adopted.

That means the `caveman` skill in your repo is already active in your Copilot too. Let's prove it.

### Try this:

1. Open **Copilot Chat** — click the chat icon in the left sidebar (or press `Ctrl+Alt+I`)
2. Ask it exactly: **"how does a python dictionary work, caveman?"**
3. Watch it respond in **caveman-speak** — short fragments, no filler words, technical terms kept intact.

The trigger word `caveman` in your question tells Copilot to load `.github/skills/caveman/SKILL.md` — the same file your notebook agent can load. Same file. Same instructions. Two different tools. One open standard.

> 💡 **Why include the word "caveman"?** Skills get picked based on their `description`. The caveman skill's description says to activate when the user says *"caveman mode"*, *"talk like caveman"*, or *"be brief"*. Drop one of those phrases into any question and watch Copilot switch modes.

*(If Copilot doesn't go caveman, it might be that GitHub Copilot Free has limited skill support — the caveman SKILL.md is still visible in the tree for everyone to read, which teaches the same lesson. Try asking the same question of your notebook agent in the next cell to see it work there.)*


## 🎯 YOUR TURN: Write your own skill

Time to add a skill. **You won't write any Python** — just create a folder and a markdown file.

### Skill ideas:

| Skill | What it does |
|-------|--------------|
| `rap-writer` | Turns any topic into a short rap verse |
| `excuse-generator` | Makes up creative excuses for missing homework |
| `code-reviewer` | Critiques code for readability and bugs |
| `shakespeare-translator` | Rewrites modern text as Shakespeare would |
| `dad-joke-teller` | Tells a dad joke related to any topic |
| `dream-interpreter` | Interprets the symbolism of a dream |
| `debate-coach` | Argues both sides of a question |
| `recipe-suggester` | Suggests a recipe using ingredients you list |

### The steps:

1. In the VS Code file tree, right-click on `.github/skills/` → **New Folder** → name it something like `dad-joke-teller`
2. Right-click your new folder → **New File** → name it `SKILL.md`
3. Copy this template into your new file and edit it:

```markdown
---
name: dad-joke-teller
description: ONE clear sentence about what this skill does and when to use it.
---

# Your Skill Name

Write the full instructions here. Be specific. Tell the agent:

1. What it should do
2. What rules to follow
3. What format to use
4. Anything it should AVOID doing
```

4. Save the file
5. Re-run the next cell below to reload skills from disk
6. Ask the agent a question that should trigger your skill

**Pro tip:** The agent only sees your `description` when deciding which skill to use. Make it clear and specific. "Writes poems" is vague — "Writes a 4-line rhyming rap verse about any topic" tells the agent exactly when to pick this skill.

In [ ]:
# Run this cell after you create your new skill file to reload skills from disk
SKILLS = load_skills_from_disk()
print(f"✅ Loaded {len(SKILLS)} skills:\n")
for name in SKILLS:
    print(f"  📖 {name}")

In [ ]:
# Now try it! Ask something that should trigger your new skill.
prompt = "REPLACE THIS with a question your skill can answer"

print(f"👤 USER: {prompt}\n")
answer = run_agent(prompt)
print(f"\n🤖 AGENT: {answer}")

## 🏆 Bonus challenges

If you finish early:

1. **The description test.** Make your skill's `description` super vague. Does the agent still pick it? Now make it super specific. What changes?

2. **Ask Copilot about your skill.** Open Copilot Chat and ask: *"Look at .github/skills/ and explain what each skill does."* Watch Copilot read your files.

3. **Skill conflict.** Add a second skill that could *also* answer your prompt. Which one does the agent pick? Why?

4. **Compound skills.** Ask a question that could use TWO skills (like "write a haiku, then translate it to pirate"). Does the agent chain them?

5. **Swap the model.** Try `MODEL = "microsoft/phi-4"` or `MODEL = "meta/llama-3.3-70b-instruct"`. Same code, different brain.

6. **Publish it.** Fork this repo, push your new skill, share the link. Anyone in the world can now use what you wrote.

## 🧠 What you just learned

You built, in under an hour, the same pattern that powers:
- **GitHub Copilot** (reads `.github/skills/`)
- **Claude Code** (reads `.claude/skills/`)
- **Microsoft Copilot Studio** (topics + knowledge sources = skills)
- **Cursor, Windsurf, Cline** (all read the same spec)

The pattern scales. Real systems have hundreds of skills. The agent still only sees the menu — it loads just the ones it needs.

### The three things to remember:

1. **A skill is just readable instructions.** Anyone can write one. You don't need to code.
2. **The agent picks skills based on descriptions.** Clear, specific descriptions win. This is a writing skill, not a coding skill.
3. **`.github/skills/` is an open standard.** The file you wrote today works in every major AI tool with zero changes.

Welcome to agentic AI. Now go write some weird skills.